# Tutorial: Entropy and Divergence for ML

- Module: 00 Math Toolkit
- Topic: information theory
- Estimated runtime: 8-10 minutes
- Prerequisites: probability distributions, expectations, basic Python, elementary plotting
- Expected outputs: entropy curves, KL divergence curves, cross-entropy/log-loss visual intuition, mutual information under noise


## Learning goals

By the end of this notebook, you should be able to:

- visualize how discrete entropy changes with distribution shape;
- compare entropy, cross-entropy, and KL divergence on simple distributions;
- connect cross-entropy to classification loss;
- inspect how Gaussian variance affects differential entropy; and
- see how noise reduces mutual information in a simple channel.


In [ ]:
from __future__ import annotations

import math

import matplotlib.pyplot as plt
import numpy as np

np.set_printoptions(precision=4, suppress=True)
plt.style.use('seaborn-v0_8-whitegrid')
SEED = 7
rng = np.random.default_rng(SEED)
SEED

## Setup helpers

The helper functions below implement the core quantities directly so the formulas stay visible. Small epsilons are used only to avoid taking a logarithm of zero on plot boundaries.


In [ ]:
EPS = 1e-9


def bernoulli_entropy(p: np.ndarray) -> np.ndarray:
    p = np.clip(np.asarray(p, dtype=float), EPS, 1.0 - EPS)
    return -(p * np.log(p) + (1.0 - p) * np.log(1.0 - p))


def bernoulli_cross_entropy(p_true: np.ndarray, q_model: np.ndarray) -> np.ndarray:
    p_true = np.clip(np.asarray(p_true, dtype=float), EPS, 1.0 - EPS)
    q_model = np.clip(np.asarray(q_model, dtype=float), EPS, 1.0 - EPS)
    return -(p_true * np.log(q_model) + (1.0 - p_true) * np.log(1.0 - q_model))


def bernoulli_kl(p_true: np.ndarray, q_model: np.ndarray) -> np.ndarray:
    return bernoulli_cross_entropy(p_true, q_model) - bernoulli_entropy(p_true)


def gaussian_differential_entropy(variance: np.ndarray) -> np.ndarray:
    variance = np.asarray(variance, dtype=float)
    return 0.5 * np.log(2.0 * np.pi * math.e * variance)


def kl_gaussian_same_variance(mu_p: float, mu_q: np.ndarray, variance: float) -> np.ndarray:
    mu_q = np.asarray(mu_q, dtype=float)
    return ((mu_p - mu_q) ** 2) / (2.0 * variance)


def binary_symmetric_channel_mi(error_rate: np.ndarray) -> np.ndarray:
    error_rate = np.clip(np.asarray(error_rate, dtype=float), EPS, 0.5)
    return math.log(2.0) - bernoulli_entropy(error_rate)

## 1. Bernoulli entropy

A Bernoulli variable is the simplest nontrivial discrete distribution. Its entropy is largest when the outcomes are balanced and smallest when one outcome is nearly certain.


In [ ]:
p_grid = np.linspace(0.001, 0.999, 400)
entropy_curve = bernoulli_entropy(p_grid)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(p_grid, entropy_curve, color='navy', lw=2)
ax.axvline(0.5, color='darkred', ls='--', lw=1.5, label='maximum at p = 0.5')
ax.set_title('Bernoulli entropy')
ax.set_xlabel('success probability p')
ax.set_ylabel('entropy H(Bernoulli(p)) [nats]')
ax.legend()
plt.show()

float(entropy_curve.max())

The peak occurs at $p=1/2$. This is the most uncertain binary distribution, so it is also the hardest binary label distribution to predict before observing any features.


## 2. Categorical entropy for a three-class distribution

For a categorical distribution, entropy is still highest when probability mass is spread evenly. The line below sweeps a one-parameter family of three-class distributions to make that idea concrete.


In [ ]:
alpha = np.linspace(0.001, 0.998, 300)
p_cat = np.stack([alpha, (1.0 - alpha) / 2.0, (1.0 - alpha) / 2.0], axis=1)
cat_entropy = -np.sum(p_cat * np.log(np.clip(p_cat, EPS, 1.0)), axis=1)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(alpha, cat_entropy, color='darkgreen', lw=2)
ax.axvline(1.0 / 3.0, color='black', ls='--', lw=1.2, label='uniform distribution')
ax.set_title('Entropy along a three-class family')
ax.set_xlabel('probability of class 1')
ax.set_ylabel('entropy [nats]')
ax.legend()
plt.show()

alpha[np.argmax(cat_entropy)], float(cat_entropy.max())

The maximum occurs near the uniform point $(1/3,1/3,1/3)$. The same pattern underlies label smoothing and uncertainty estimates in multiclass settings.


## 3. Cross-entropy and KL divergence for Bernoulli models

Fix a true Bernoulli distribution with parameter $p=0.8$ and vary the model parameter $q$. Cross-entropy measures expected log loss under the model, while KL divergence measures the excess over the true entropy.


In [ ]:
p_true = 0.8
q_grid = np.linspace(0.001, 0.999, 400)
true_entropy = bernoulli_entropy(p_true)
cross_entropy_curve = bernoulli_cross_entropy(p_true, q_grid)
kl_curve = bernoulli_kl(p_true, q_grid)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(q_grid, cross_entropy_curve, label='cross-entropy H(P, Q)', color='purple', lw=2)
ax.plot(q_grid, kl_curve, label='KL divergence D_KL(P || Q)', color='darkorange', lw=2)
ax.axvline(p_true, color='black', ls='--', lw=1.2, label='q = p')
ax.axhline(float(true_entropy), color='gray', ls=':', lw=1.5, label='true entropy H(P)')
ax.set_title('Cross-entropy decomposition for Bernoulli distributions')
ax.set_xlabel('model parameter q')
ax.set_ylabel('nats')
ax.legend()
plt.show()

{
    'true_entropy': float(true_entropy),
    'min_cross_entropy': float(cross_entropy_curve.min()),
    'min_kl': float(kl_curve.min()),
}

Both curves are minimized when the model matches the truth. The vertical gap between cross-entropy and entropy is exactly the KL divergence, so empirical cross-entropy training is a practical way to push the model distribution toward the target distribution.


## 4. Cross-entropy as classification loss

For one-hot labels, cross-entropy reduces to negative log-probability of the correct class. The plot below shows how sharply the loss grows when the model assigns low probability to the true class.


In [ ]:
correct_class_prob = np.linspace(0.001, 0.999, 400)
classification_loss = -np.log(correct_class_prob)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(correct_class_prob, classification_loss, color='crimson', lw=2)
ax.set_title('Per-example cross-entropy for the correct class')
ax.set_xlabel('predicted probability of the true class')
ax.set_ylabel('loss = -log p_true')
plt.show()

for p in [0.9, 0.6, 0.1, 0.01]:
    print(f'p_true={p:>4}: loss={-math.log(p):.4f}')

This is the standard classification objective used with softmax outputs. Confident correct predictions receive small loss; confident mistakes receive very large loss.


## 5. Differential entropy of a Gaussian

For a Gaussian distribution, differential entropy depends only on variance. A wider Gaussian spreads mass over a larger region, which raises differential entropy.


In [ ]:
variance_grid = np.linspace(0.05, 4.0, 300)
gaussian_entropy = gaussian_differential_entropy(variance_grid)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(variance_grid, gaussian_entropy, color='teal', lw=2)
ax.set_title('Differential entropy of N(0, variance)')
ax.set_xlabel('variance')
ax.set_ylabel('h(X) [nats]')
plt.show()

{
    'h(var=0.25)': float(gaussian_differential_entropy(np.array([0.25]))[0]),
    'h(var=1.0)': float(gaussian_differential_entropy(np.array([1.0]))[0]),
    'h(var=4.0)': float(gaussian_differential_entropy(np.array([4.0]))[0]),
}

Unlike discrete entropy, differential entropy can be negative for very concentrated continuous distributions. That is one reason KL divergence and mutual information are usually easier to interpret robustly.


## 6. KL divergence for Gaussians with equal variance

When two one-dimensional Gaussians share variance $\sigma^2$, the KL divergence depends quadratically on the difference in means:

$$
D_{KL}(\mathcal{N}(\mu_p, \sigma^2)\|\mathcal{N}(\mu_q, \sigma^2)) = rac{(\mu_p - \mu_q)^2}{2\sigma^2}.
$$

This is a useful warmup before the more general Gaussian KL formulas used in latent-variable models.


In [ ]:
mu_p = 0.0
mu_q_grid = np.linspace(-3.0, 3.0, 300)
variance = 1.0
gaussian_kl_curve = kl_gaussian_same_variance(mu_p, mu_q_grid, variance)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(mu_q_grid, gaussian_kl_curve, color='brown', lw=2)
ax.axvline(mu_p, color='black', ls='--', lw=1.2, label='matching mean')
ax.set_title('KL divergence between Gaussians with equal variance')
ax.set_xlabel('model mean')
ax.set_ylabel('D_KL(P || Q)')
ax.legend()
plt.show()

float(kl_gaussian_same_variance(0.0, np.array([2.0]), 1.0)[0])

The minimum occurs when the two Gaussians match. In VAE objectives, KL terms play this same role: they penalize posterior distributions that drift away from a chosen prior.


## 7. Mutual information in a noisy binary channel

Let $X \sim \mathrm{Bernoulli}(1/2)$ and let $Y$ be formed by flipping $X$ with probability $arepsilon$. Then

$$
I(X;Y) = \log 2 - H(\mathrm{Bernoulli}(arepsilon)).
$$

As the channel becomes noisier, the observation $Y$ reveals less about the source $X$.


In [ ]:
error_grid = np.linspace(0.001, 0.5, 300)
mi_curve = binary_symmetric_channel_mi(error_grid)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(error_grid, mi_curve, color='darkslateblue', lw=2)
ax.set_title('Mutual information in a binary symmetric channel')
ax.set_xlabel('flip probability epsilon')
ax.set_ylabel('I(X; Y) [nats]')
plt.show()

{
    'MI at epsilon=0.0 (limit)': float(math.log(2.0)),
    'MI at epsilon=0.1': float(binary_symmetric_channel_mi(np.array([0.1]))[0]),
    'MI at epsilon=0.5': float(binary_symmetric_channel_mi(np.array([0.5]))[0]),
}

This is a direct picture of the data processing intuition. Additional noise removes recoverable information about the original source. A downstream model can reorganize the remaining signal, but it cannot reconstruct information that the channel has destroyed.


## Pitfall and extension

A common mistake is to treat entropy, KL divergence, and mutual information as interchangeable notions of uncertainty. They answer different questions:

- entropy measures uncertainty under one distribution;
- cross-entropy and KL compare two distributions; and
- mutual information measures dependence between variables.

Extension: modify the notebook to compare reverse KL and forward KL on the same Bernoulli family, then write down how the asymmetry changes what errors are penalized most strongly.


## Summary

The notebook illustrated the quantities that recur throughout later ML modules:

- entropy peaks when a distribution is most spread out over its support;
- cross-entropy becomes classification log loss when labels are one-hot;
- KL divergence measures the extra coding or log-loss cost of using the wrong distribution;
- Gaussian variance raises differential entropy; and
- mutual information falls as a noisy channel destroys signal.
